# 01 · Hybrid matching: job-market skills vs. AUA syllabus topics

This notebook runs the matching model at the centre of the thesis and produces
`results/skill_coverage.csv`.

**Inputs** (not included in the repo, see `data/README.md`):

| file | what it holds |
|---|---|
| `top_canonical_skills.csv` | 170 high-frequency canonical skills with job counts and domain |
| `syllabi_topics_canonicalized.csv` | canonical topics for each of the 77 AUA syllabi |
| `reviewed_gap_types.csv` | the gap type I assigned to each skill during the review step |

**Steps:** build a topic index → lexical match (own department, then the other
department) → SBERT similarity for everything still unmatched → apply the
review layer → save.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from skillgap.matching import HybridMatcher, build_course_index, results_to_frame, SBERT_MODEL, SBERT_THRESHOLD
from skillgap.validation import apply_validation

DATA = Path("../data/private")
RESULTS = Path("../results")
pd.set_option("display.width", 140, "display.max_colwidth", 60)

## Load skills and syllabi

In [2]:
skills = pd.read_csv(DATA / "top_canonical_skills.csv")[["skill", "jobs", "pct", "domain"]]
syllabi = pd.read_csv(DATA / "syllabi_topics_canonicalized.csv")
syllabi = syllabi[syllabi["source_dataset"].isin(["CSE", "BUS"])]

print(f"{len(skills)} candidate skills | {len(syllabi)} syllabi")
print(skills["domain"].value_counts().to_string())
skills.head(10)

170 candidate skills | 77 syllabi
domain
BUS        89
CSE        74
CSE+BUS     7


,skill,jobs,pct,domain
0,Microsoft Office,365,22.7,BUS
1,Excel,253,15.7,CSE+BUS
2,Negotiation,252,15.7,BUS
3,Customer Service,227,14.1,BUS
4,Sales,170,10.6,BUS
5,CRM,145,9.0,BUS
6,Microsoft Word,127,7.9,BUS
7,Accounting,105,6.5,BUS
8,Market Research,104,6.5,BUS
9,Data Analysis,97,6.0,CSE+BUS


## Build the topic index

Each course keeps its canonical topics plus a token set per topic for the
lexical check. Two syllabi have no course code; keying on the blank code would
make the second silently overwrite the first, so they're keyed by title
instead.

In [3]:
courses = build_course_index(syllabi)
matcher = HybridMatcher(courses, threshold=SBERT_THRESHOLD)

print(f"courses indexed : {len(courses)}")
print(f"unique topics   : {len(matcher.topics)}")
print(f"SBERT model     : {SBERT_MODEL}, threshold {SBERT_THRESHOLD}")

courses indexed : 77
unique topics   : 853
SBERT model     : all-MiniLM-L12-v2, threshold 0.6


## Run the matcher

For every skill: (1) lexical match in its own department, (2) lexical match in
the other department, (3) SBERT cosine similarity. The first step that finds
something wins.

In [4]:
matches = results_to_frame(matcher.match(skills), skills)

step_names = {1: "lexical, own department", 2: "lexical, other department", 3: f"SBERT (>= {SBERT_THRESHOLD})"}
summary = (matches.groupby("step")["status"].value_counts().unstack(fill_value=0)
                  .rename(index=step_names))
print(summary.to_string())
print(f"\ncovered: {(matches.status == 'COVERED').sum()} / {len(matches)}")

status                     COVERED  MISSING
step                                       
lexical, own department         62        0
lexical, other department        8        0
SBERT (>= 0.6)                  40       60

covered: 110 / 170


## Final target set (165 skills)

Two adjustments from the thesis before the final comparison:

* five labels were removed in the final filtering for curriculum
  comparability ("Finance", "Economics", "Data Science", "Data Engineering",
  "Contract Negotiation"): they name a whole field or overlap with a more
  specific skill already in the list;
* the seven skills tagged as cross-domain were each assigned to CBE or CSE,
  since a separate group of seven would make percentages unstable.

`BUS` in the syllabus data is the College of Business and Economics, reported
as **CBE** from here on.

In [5]:
reviewed = pd.read_csv(DATA / "reviewed_gap_types.csv")

final = (matches.merge(reviewed, on="skill", how="inner")
                .drop(columns="domain").rename(columns={"final_domain": "domain"}))
dropped = sorted(set(matches.skill) - set(final.skill))

print(f"dropped from target set: {dropped}")
print(f"final skills: {len(final)}  ->  {final.domain.value_counts().to_dict()}")
print(f"model-based coverage: {(final.status == 'COVERED').sum()} covered, "
      f"{(final.status == 'MISSING').sum()} missing "
      f"({(final.status == 'COVERED').mean():.1%})")

dropped from target set: ['Contract Negotiation', 'Data Engineering', 'Data Science', 'Economics', 'Finance']
final skills: 165  ->  {'CBE': 92, 'CSE': 73}
model-based coverage: 106 covered, 59 missing (64.2%)


## Review layer: covered / partial / missing

The binary model can't tell "taught" from "related topic exists" or "taught
but not written in the syllabus". Each result was reviewed (an LLM read the
candidate courses' syllabus text and proposed a status with a reason, then I
checked it) and given one of four gap types. The final status follows from the
gap type.

In [6]:
final = apply_validation(final, gap_col="gap_type")

print(pd.crosstab(final["status"], final["final_status"], margins=True).to_string())
print()
print(pd.crosstab(final["domain"], final["final_status"], margins=True)[["COVERED", "PARTIAL", "MISSING", "All"]].to_string())

final_status  COVERED  MISSING  PARTIAL  All
status                                      
COVERED            97        2        7  106
MISSING             0       27       32   59
All                97       29       39  165

final_status  COVERED  PARTIAL  MISSING  All
domain                                      
CBE                70       12       10   92
CSE                27       27       19   73
All                97       39       29  165


## Save `results/skill_coverage.csv`

The public table keeps skill-level results only. Syllabus topics, course names
and the text the reviewer read stay private.

In [7]:
out = final.rename(columns={"jobs": "n_jobs", "pct": "pct_of_postings", "status": "model_status",
                            "score": "similarity_score", "step": "match_step"})
out = out[["skill", "domain", "n_jobs", "pct_of_postings", "match_step", "model_status",
           "similarity_score", "gap_type", "final_status"]]
out = out.sort_values("n_jobs", ascending=False).reset_index(drop=True)

RESULTS.mkdir(exist_ok=True)
out.to_csv(RESULTS / "skill_coverage.csv", index=False)
print(f"saved {len(out)} rows to results/skill_coverage.csv")
out.head(15)

saved 165 rows to results/skill_coverage.csv


,skill,domain,n_jobs,pct_of_postings,match_step,model_status,similarity_score,gap_type,final_status
0,Microsoft Office,CBE,365,22.7,3,MISSING,0.4614,Teaching Gap,MISSING
1,Excel,CBE,253,15.7,3,MISSING,0.4911,Teaching Gap,MISSING
2,Negotiation,CBE,252,15.7,3,MISSING,0.4818,Teaching Gap,MISSING
3,Customer Service,CBE,227,14.1,1,COVERED,1.0000,Lexical Coverage,COVERED
4,Sales,CBE,170,10.6,1,COVERED,1.0000,Lexical Coverage,COVERED
5,CRM,CBE,145,9.0,3,MISSING,0.5844,Partial / Documentation Gap,PARTIAL
6,Microsoft Word,CBE,127,7.9,3,MISSING,0.4356,Teaching Gap,MISSING
7,Accounting,CBE,105,6.5,1,COVERED,1.0000,Lexical Coverage,COVERED
8,Market Research,CBE,104,6.5,1,COVERED,1.0000,Lexical Coverage,COVERED
9,Data Analysis,CBE,97,6.0,1,COVERED,1.0000,Lexical Coverage,COVERED
